Dear TAs, I was running this notebook on Kaggle Code.
If you are running in google colab, please modify some path in this file
And to uncomment the pip install in the first cell, also following some annotations



In [ ]:
## Cell 1: Environment Setup

# If using colab, uncomment the following line to install necessary packages
# !pip install tensorflow pandas numpy scikit-learn pathlib

# This two lines are for some env problems in Kaggle
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"
print("Environment variables configured.")

In [ ]:
## Cell 2: Import Libraries

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
# from google.colab import drive, files # Colab specific tools


print("TensorFlow:", tf.__version__)
print("All libraries imported successfully.")

In [ ]:
## Cell 2.5: Enable Mixed Precision

# Set the global dtype policy to 'mixed_float16'
# This will convert most computations (like convolutions) to float16 (half precision) to speed up training on GPUs.
# The output layer (like softmax) still uses float32 to maintain numerical stability.
policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)

print(f"Global policy set to: {policy.name}")
print("This should speed up training on compatible GPUs. (like T4, V100, A100)")

In [ ]:
## Cell 3: Import Data and Set Paths
## Please modify the paths according to your folder structure. If you are using Colab.

# 1. Mount Google Drive
# drive.mount('/content/drive')
# print("Google Drive mounted.")

# 2. Set data paths

# **Please modify the paths according to your extracted folder structure**
# Assuming the extracted folders for training and testing data are structured as: DATA_ROOT/train and DATA_ROOT/test
DATA_ROOT = Path("/kaggle/input/nycu-114-1-introduction-to-machine-learning-hw-4/data")

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

# 3. Some hyperparameters
IMG_SIZE = 96      # image size (96x96)
BATCH_SIZE = 32    # batch size
AUTOTUNE = tf.data.AUTOTUNE # TensorFlow data performance optimization

print(f"Train directory: {TRAIN_DIR}")
print(f"Test directory: {TEST_DIR}")

In [ ]:
## Cell 4: Prepare DataFrame from Folder Structure

def df_from_folder(root):
    rows = []
    root = Path(root)
    for c in sorted(root.iterdir()):
        if c.is_dir():
            label = c.name
            for p in c.glob("*"):
                if p.suffix.lower() in (".jpg", ".jpeg", ".png"):
                    rows.append({
                        "filepath": str(p),
                        "label": label
                    })
    return pd.DataFrame(rows)

if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"Train directory not found at: {TRAIN_DIR}. Please check Cell 3.")

df = df_from_folder(TRAIN_DIR)

# Convert labels: original labels (text) -> numeric codes
df["label_orig"] = df["label"]
df["label"] = pd.Categorical(df["label"]).codes

# Create label mapping (numeric code <-> original label name)
label_map = dict(enumerate(pd.Categorical(df["label_orig"]).categories))
NUM_CLASSES = len(label_map)

print(f"Total training images: {len(df)}")
print(f"Number of classes (NUM_CLASSES): {NUM_CLASSES}")
print(f"Label mapping: {label_map}")
print("\nDataFrame Head:")
print(df.head())

In [ ]:
## Cell 5: Dataset Splitting, Processing Functions, and Dataset Creation

# 1. Train-validation split
# Use stratify to ensure similar class distribution in train and validation sets
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42
)
print(f"訓練集大小: {len(train_df)}")
print(f"驗證集大小: {len(val_df)}")

# 2. parse the pictures
def parse(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = image / 255.0
    return image, tf.cast(label, tf.int32)

# 3. Data augmentation function
def augment(image, label):
    image = tf.image.random_flip_left_right(image) # Random horizontal flip
    return image, label

# 4. Create tf.data.Dataset function
def make_ds(df, batch=BATCH_SIZE, shuffle=True, aug=False):
    paths = df["filepath"].values
    labels = df["label"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(2000)
    ds = ds.map(parse, AUTOTUNE)
    if aug:
        ds = ds.map(augment, AUTOTUNE)
    # Batch and prefetch for improved training efficiency
    ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, aug=True)
val_ds = make_ds(val_df, shuffle=False)

print("Training and Validation Datasets ready.")

In [ ]:
## Cell 6: The Model Structure and Compilation (Enhanced ResNet)

# 1. Helper function: Convolutional Block (Conv Block)
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

# 2. Helper function: Residual Block (Res Block)
def res_block(x, filters):
    shortcut = x
    x = conv_block(x, filters)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    # Ensure the input and output shapes of the residual connection are the same.
    x = layers.add([shortcut, x])
    return layers.ReLU()(x)

# 3. Model definition
inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))

x = conv_block(inputs, 32)
x = layers.MaxPooling2D()(x)

x = conv_block(x, 64)
x = res_block(x, 64)
x = res_block(x, 64) # added an extra residual block
x = layers.MaxPooling2D()(x)

x = conv_block(x, 128)
x = res_block(x, 128)
x = res_block(x, 128) # added an extra residual block
x = layers.MaxPooling2D()(x) # added an extra pooling layer

x = conv_block(x, 256) # added a higher-dimensional convolutional layer
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.5)(x) # increased Dropout to 0.5

# Output layer still uses float32 for numerical stability (with Mixed Precision)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype='float32')(x)

model = keras.Model(inputs, outputs)

# 4. Model compilation
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 5. Model summary
model.summary()

In [ ]:
## Cell 7: Model Training with Callbacks

# 1. Set up Callbacks
# EarlyStopping: Stop training if val_loss does not improve for 5 consecutive epochs.
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True # Restore weights from the epoch with the best validation loss
)

# ReduceLROnPlateau: If val_loss does not improve for 3 consecutive epochs, reduce the learning rate by 50%.
lr_schedule = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

EPOCHS = 30 # increased max epochs, relying on Early Stopping to decide the actual stopping point
print(f"Starting model training (Max {EPOCHS} epochs) with Early Stopping...")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, lr_schedule]
)
print("Model training finished.")

In [ ]:
## Cell 8: Prediction on Test Set and Save Results

# 1. Load test image paths
if not TEST_DIR.exists():
    raise FileNotFoundError(f"Test directory not found at: {TEST_DIR}. Please check Cell 3.")

test_paths = sorted(TEST_DIR.glob("*"))
test_files = [str(p) for p in test_paths if p.suffix.lower() in (".jpg",".jpeg",".png")]
print(f"Found {len(test_files)} test images.")

# 2. load test images
def load_test(path):
    x = tf.io.read_file(path)
    x = tf.image.decode_jpeg(x,3)
    x = tf.image.resize(x,[IMG_SIZE,IMG_SIZE]) / 255. # Resize and normalize
    return x

# 3. Load all test images and stack into a Tensor
X_test = tf.stack([load_test(p) for p in test_files])

# 4. Predict on test data
print("Predicting on test data...")
pred = model.predict(X_test)

# 5. Process prediction results
# Find the numeric code of the class with the highest probability
pred_codes = np.argmax(pred, axis=1)
# Convert numeric codes back to original labels (text)
pred_labels = [label_map[int(code)] for code in pred_codes]

# 6. Convert to numeric labels  (generated=1, others=0)
pred_labels_numeric = [1 if label == "generated" else 0 for label in pred_labels]
df_pred = pd.DataFrame({
    "filename": [Path(p).stem for p in test_files],
    "label": pred_labels_numeric
})

# 7. Save prediction results to CSV
OUTPUT_PATH = "/kaggle/working/prediction_results.csv"
df_pred.to_csv(OUTPUT_PATH, index=False)
print(f"Prediction results saved to {OUTPUT_PATH}")

print("\nPrediction results head:")
print(df_pred.head())

In [ ]:
## Cell X: Save Model Weights to Output

import shutil
from pathlib import Path

MODEL_WEIGHTS_FILENAME = "image_classification_model.weights.h5"

OUTPUT_PATH = Path("/kaggle/working") / MODEL_WEIGHTS_FILENAME

try:
    model.save_weights(OUTPUT_PATH)
    print(f"✅ Success:")
    print(f"Model weights saved successfully to Kaggle Output as: {OUTPUT_PATH}")
    print("-" * 50)
except Exception as e:
    print(f" Error saving model weights: {e}")
    print("-" * 50)
